# OpenBind-HIPPO

- **Target: D68EV3C**
- **Cycle: 01**

## Imports

In [2]:
%load_ext autoreload
%autoreload 2
import hippo
import mrich
from mrich import print
from pathlib import Path
from os import environ
import shutil
import molparse as mp
import pandas as pd
import plotly.express as px

## Config

In [3]:
target_name = "D68EV3C"
target_dir = Path(environ["BULK"]) / "TARGETS" / target_name
cycle_name = "cycle_01"
cycle_dir = Path(cycle_name)
aligned_dir = target_dir / "aligned_files"

## Animal

In [4]:
animal = hippo.HIPPO(target_name, target_dir / f"{target_name}.sqlite")

 Creating HIPPO animal

name = D68EV3C

db_path = /opt/xchem-fragalysis-2/maxwin/BulkDock/TARGETS/D68EV3C/D68EV3C.sqlite

DEBUG: hippo.Database.__init__()

DEBUG: Database.path = /opt/xchem-fragalysis-2/maxwin/BulkDock/TARGETS/D68EV3C/D68EV3C.sqlite

DEBUG: hippo.Database.connect()

DEBUG: sqlite3.version='2.6.0'

 Success  Database connected @ /opt/xchem-fragalysis-2/maxwin/BulkDock/TARGETS/D68EV3C/D68EV3C.sqlite!

 Success  Initialised animal HIPPO("D68EV3C")!

## Queue BulkDock (re)placements

In [13]:
scaffolds = animal.compounds(tag="openbind_d68ev3c_c1_scaffolds_chemok_bbok")
elab_poses = scaffolds.elabs.poses
scaffolds, elab_poses

(compounds tagged openbind_d68ev3c_c1_scaffolds_chemok_bbok: {C × 77},
 {P × 231})

In [20]:
dedupe = set()
for pose in mrich.track(elab_poses):
    compound = pose.compound
    scaffolds = compound.scaffolds
    if len(scaffolds) != 1:
        mrich.warning("multiple", pose)
    scaffold = scaffolds[0]
    
    reference = pose.reference
    inspirations = pose.inspirations

    dedupe.add((scaffold.id, reference.id, tuple(inspirations.ids)))

    # elabs = scaffold.elabs

data = []

for scaffold_id, reference_id, inspiration_ids in dedupe:
    print(scaffold_id, reference_id, inspiration_ids)

    scaffold = animal.compounds[scaffold_id]
    reference = animal.poses[reference_id]
    inspirations = animal.poses[inspiration_ids]

    inspiration_d = dict()
    for i, name in enumerate(inspirations.names):
        inspiration_d[f"hit{i+1}"] = name
    
    for elab in mrich.track(scaffold.elabs):
        d = dict(smiles=elab.smiles)
        d.update(inspiration_d)
        data.append(d)

df = pd.DataFrame(data)
df.head()

Output()

33797 97
(44, 59)

Output()

30265 55
(106, 124)

Output()

46521 99
(98, 111)

Output()

,smiles,hit1,hit2
0,Cc1nn(CCNC(=O)CCc2c(C)[nH]c3ccccc23)cc1Cl,7gp2-a,7goy-a
1,Cc1[nH]c2ccccc2c1CCC(=O)NCCn1ncc(Cl)c1C,7gp2-a,7goy-a
2,Cc1c(Cl)cnn1CCNC(=O)CCc1cn(C)c2ccccc12,7gp2-a,7goy-a
3,Cc1nn(CCNC(=O)CCc2cn(C)c3ccccc23)cc1Cl,7gp2-a,7goy-a
4,Cc1ccc2[nH]cc(CCC(=O)NCCn3cc(Cl)c(C)n3)c2c1,7gp2-a,7goy-a


In [21]:
print(len(df))
df = df.drop_duplicates()
print(len(df))

1170

1170

In [22]:
df.to_csv("cycle_01/syndirella/elabs/d68ev3c_c1_elab_bulkdock_input.csv", index=False)
df

,smiles,hit1,hit2
0,Cc1nn(CCNC(=O)CCc2c(C)[nH]c3ccccc23)cc1Cl,7gp2-a,7goy-a
1,Cc1[nH]c2ccccc2c1CCC(=O)NCCn1ncc(Cl)c1C,7gp2-a,7goy-a
2,Cc1c(Cl)cnn1CCNC(=O)CCc1cn(C)c2ccccc12,7gp2-a,7goy-a
3,Cc1nn(CCNC(=O)CCc2cn(C)c3ccccc23)cc1Cl,7gp2-a,7goy-a
4,Cc1ccc2[nH]cc(CCC(=O)NCCn3cc(Cl)c(C)n3)c2c1,7gp2-a,7goy-a
...,...,...,...
1165,O=C(NC(Cc1c[nH]c2ccccc12)C(=O)N1CCC(n2cc(Cl)cn...,7gp2-a,7goy-a
1166,CCC(CNC(=O)C(Cc1c[nH]c2ccccc12)NC(=O)c1ccc(F)c...,7gp2-a,7goy-a
1167,Cc1nn(CCNC(=O)C(Cc2c[nH]c3ccccc23)NC(=O)c2ccc(...,7gp2-a,7goy-a
1168,COC(=O)C(Cc1cc[nH]n1)NC(=O)C1Cc2[nH]ncc2C(=O)N1,7go4-a,7gpo-a


In [11]:
# scaffold_poses = animal.poses(tag="openbind_d68ev3c_c1_scaffolds_chemok_bbok")
# elaborated_scaffolds = scaffold_poses.compounds.elabs.scaffolds
# scaffold_poses.compounds.elabs
# elaborated_scaffold_poses = animal.poses[set(elaborated_scaffolds.poses.ids).intersection(set(scaffold_poses.ids))]
# elaborated_scaffold_poses

## Define chemspace

In [5]:
scaffolds = animal.compounds(tag="openbind_d68ev3c_c1_scaffolds_chemok_bbok")
scaffolds

compounds tagged openbind_d68ev3c_c1_scaffolds_chemok_bbok: {C × 77}

In [6]:
elabs = scaffolds.elabs
elabs

{C × 1211}

In [7]:
posed_elabs = elabs.poses.compounds
posed_elabs

{C × 1170}

In [8]:
posed_elabs.add_tag("openbind_d68ev3c_c1_elabs")

Tagged {C × 1170} w/ "openbind_d68ev3c_c1_elabs"

In [9]:
posed_elabs.scaffolds.add_tag("openbind_d68ev3c_c1_elaborated_scaffolds")

Tagged {C × 3} w/ "openbind_d68ev3c_c1_elaborated_scaffolds"

In [10]:
chemspace = posed_elabs + posed_elabs.scaffolds
chemspace

{C × 1173}

In [11]:
full_recipe = hippo.Recipe.from_compounds(chemspace)

#compounds = 1173

Output()

Solving recipe combinations...

Output()

DEBUG: Calculating prices...

Picking cheapest from 1 options

In [ ]:
full_recipe.write_json("cycle_01/syndirella/elabs/openbind_d68ev3c_c1_elab_chemspace.json")

In [ ]:
full_recipe.write_CAR_csv("cycle_01/syndirella/elabs/openbind_d68ev3c_c1_elab_chemspace.csv")

In [ ]:
full_recipe.write_reactant_csv("cycle_01/syndirella/elabs/openbind_d68ev3c_c1_elab_chemspace_reactants.csv")

In [4]:
animal.tags.summary();

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃ tag                                       ┃ num_compounds ┃ num_poses ┃ num_posed_compounds ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ BAD count                                 │ 0             │ 202       │ 156                 │
│ BulkDock Fragalysis export                │ 0             │ 351       │ 351                 │
│ GOOD count                                │ 0             │ 202       │ 156                 │
│ MEDIOCRE count                            │ 0             │ 202       │ 156                 │
│ Main status                               │ 0             │ 202       │ 156                 │
│ [Other] iter1_frags                       │ 0             │ 42        │ 42                  │
│ [Other] upload_1 2025-02-11               │ 0             │ 202       │ 156                 │
│ d68ev3c_c1_elab_bulkdock_input            │ 1170          │ 2340      │ 1170                │
│ fragmenstein_placed                       │ 0             │ 63108     │ 42916               │
│ hits                                      │ 156           │ 202       │ 156                 │
│ openbind_d68ev3c_c1_elaborated_scaffolds  │ 3             │ 0         │ 0                   │
│ openbind_d68ev3c_c1_elabs                 │ 157           │ 0         │ 0                   │
│ openbind_d68ev3c_c1_fragmenstein          │ 1665          │ 2003      │ 1665                │
│ openbind_d68ev3c_c1_knitwork_impure       │ 18230         │ 20662     │ 15107               │
│ openbind_d68ev3c_c1_knitwork_pure         │ 25521         │ 38103     │ 25473               │
│ openbind_d68ev3c_c1_scaffolds             │ 345           │ 345       │ 345                 │
│ openbind_d68ev3c_c1_scaffolds_chemok      │ 175           │ 0         │ 0                   │
│ openbind_d68ev3c_c1_scaffolds_chemok_bbok │ 77            │ 77        │ 77                  │
└───────────────────────────────────────────┴───────────────┴───────────┴─────────────────────┘

In [11]:
animal.scaffolds.elabs.poses.compounds

{C × 1170}

In [12]:
animal.elabs.poses.compounds.scaffolds

{C × 3}